# BÀI THỰC HÀNH 04: HỒI QUY TUYẾN TÍNH & HỒI QUY LOGISTIC
## BÀI TOÁN 1: HỒI QUY TUYẾN TÍNH – DỰ ĐOÁN GIÁ BẤT ĐỘNG SẢN (HOUSING PRICES)

* **Môn học:** Học máy (Machine Learning)
* **Sinh viên thực hiện:** Nguyễn Đức Duy
* **Mã số sinh viên (MSSV):** 65130628
* **Lớp:** Trí tuệ Nhân tạo - KHMT
* **Môi trường:** Python 3.10+, JupyterLab, Scikit-learn, Pandas, NumPy, Matplotlib, Seaborn

---
### I. BỐI CẢNH BÀI TOÁN & CƠ SỞ TOÁN HỌC
Định giá bất động sản là một bài toán cốt lõi trong phân tích thị trường tài chính và kinh tế lượng. Giá trị một ngôi nhà chịu tác động đồng thời bởi nhiều yếu tố: quy mô vật lý (diện tích sàn, số phòng ngủ, số tầng), vị trí (tiếp giáp mặt đường chính, khu vực dân cư ưu tiên) và mức độ tiện nghi (điều hòa, bình nước nóng, tầng hầm).

Mục tiêu là xây dựng mô hình **Hồi quy tuyến tính đa biến (Multiple Linear Regression)** nhằm ước lượng biến mục tiêu liên tục `price` (đơn vị USD):
$$\hat{y} = w_0 + w_1 x_1 + w_2 x_2 + \dots + w_d x_d$$

Trong đó các trọng số $w$ được tối ưu sao cho hàm mất mát sai số toàn phương trung bình (MSE) đạt giá trị nhỏ nhất:
$$J(w) = \frac{1}{N} \sum_{i=1}^N (y_i - \hat{y}_i)^2$$

Mô hình Scikit-learn giải phương trình nghiệm giải tích (Normal Equation):
$$w = (X^T X)^{-1} X^T y$$


### Khối 1.1: Đọc dữ liệu và bóc tách biến mục tiêu
* **Thực hiện:** Tải dữ liệu từ thư mục `../data/raw/Housing.csv`, khám phá cấu trúc tập dữ liệu và tách riêng ma trận đặc trưng $X_{raw}$ và vector mục tiêu $y$.
* **Ý nghĩa:** Tránh để biến mục tiêu lọt vào tập đầu vào khiến mô hình học trực tiếp đáp án (Target Leakage).

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Cấu hình giao diện biểu đồ
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
%matplotlib inline

# Nạp dữ liệu từ thư mục raw
df_housing = pd.read_csv('../data/raw/Housing.csv')

print(f"Kích thước tập dữ liệu thô: {df_housing.shape[0]} dòng, {df_housing.shape[1]} cột")
display(df_housing.head())

# Tách ma trận đặc trưng X và vector mục tiêu y
X_raw = df_housing.drop(columns=['price'])
y = df_housing['price']

print("\nThông tin tóm tắt các kiểu dữ liệu:")
print(X_raw.dtypes.value_counts())


Kích thước tập dữ liệu thô: 545 dòng, 13 cột


,price,area,bedrooms,bathrooms,stories,mainroad,guestroom,basement,hotwaterheating,airconditioning,parking,prefarea,furnishingstatus
0,13300000,7420,4,2,3,yes,no,no,no,yes,2,yes,furnished
1,12250000,8960,4,4,4,yes,no,no,no,yes,3,no,furnished
2,12250000,9960,3,2,2,yes,no,yes,no,no,2,yes,semi-furnished
3,12215000,7500,4,2,2,yes,no,yes,no,yes,3,yes,furnished
4,11410000,7420,4,1,2,yes,yes,yes,no,yes,2,no,furnished



Thông tin tóm tắt các kiểu dữ liệu:
str      7
int64    5
Name: count, dtype: int64


### Khối 1.2: Số hóa các biến định tính (Categorical Encoding)
* **Thực hiện:** Chuyển đổi định dạng văn bản thành các giá trị số. Biến nhị phân chuyển thành $\{0, 1\}$; biến `furnishingstatus` được tạo thành các cột chỉ báo mới thông qua kỹ thuật One-Hot Encoding.
* **Ý nghĩa kỹ thuật:** Thuật toán hồi quy yêu cầu đầu vào là đại số số học. Tùy chọn `drop_first=True` giúp loại bỏ một cột cơ sở, ngăn chặn bẫy biến giả (**Dummy Variable Trap**) dẫn đến hiện tượng đa cộng tuyến hoàn hảo (**Multicollinearity**), làm ma trận $X^T X$ suy biến không khả nghịch.

In [2]:
# 1. Chuyển đổi các cột nhị phân text sang số 0 và 1
binary_cols = ['mainroad', 'guestroom', 'basement', 'hotwaterheating', 'airconditioning', 'prefarea']
for col in binary_cols:
    X_raw[col] = X_raw[col].map({'yes': 1, 'no': 0})

# 2. Áp dụng One-Hot Encoding cho biến đa phân loại furnishingstatus
X_encoded = pd.get_dummies(X_raw, columns=['furnishingstatus'], drop_first=True, dtype=int)

print(f"Kích thước ma trận sau mã hóa: {X_encoded.shape}")
print("Danh sách các cột đặc trưng sau mã hóa:")
print(list(X_encoded.columns))
display(X_encoded.head())


Kích thước ma trận sau mã hóa: (545, 13)
Danh sách các cột đặc trưng sau mã hóa:
['area', 'bedrooms', 'bathrooms', 'stories', 'mainroad', 'guestroom', 'basement', 'hotwaterheating', 'airconditioning', 'parking', 'prefarea', 'furnishingstatus_semi-furnished', 'furnishingstatus_unfurnished']


,area,bedrooms,bathrooms,stories,mainroad,guestroom,basement,hotwaterheating,airconditioning,parking,prefarea,furnishingstatus_semi-furnished,furnishingstatus_unfurnished
0,7420,4,2,3,1,0,0,0,1,2,1,0,0
1,8960,4,4,4,1,0,0,0,1,3,0,0,0
2,9960,3,2,2,1,0,1,0,0,2,1,1,0
3,7500,4,2,2,1,0,1,0,1,3,1,0,0
4,7420,4,1,2,1,1,1,0,1,2,0,0,0


### Khối 1.3: Phân chia tập dữ liệu huấn luyện và kiểm thử (Train/Test Split)
* **Thực hiện:** Phân chia ngẫu nhiên tập dữ liệu thành 80% cho huấn luyện (Train) và 20% cho đánh giá độc lập (Test).
* **Ý nghĩa:** Cố định `random_state=42` giúp thực nghiệm có tính tái lập (reproducibility). Tập Test mô phỏng các căn nhà mới chưa từng xuất hiện trên thị trường để đánh giá năng lực tổng quát hóa.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_encoded, y, test_size=0.2, random_state=42
)

print(f"Số lượng mẫu huấn luyện (Train): {X_train.shape[0]} ({X_train.shape[0]/len(X_encoded)*100:.1f}%)")
print(f"Số lượng mẫu kiểm thử (Test):     {X_test.shape[0]} ({X_test.shape[0]/len(X_encoded)*100:.1f}%)")


### Khối 1.4: Chuẩn hóa đặc trưng & Phòng chống rò rỉ dữ liệu (Feature Scaling)
* **Thực hiện:** Đưa các biến số có khoảng giá trị chênh lệch (ví dụ `area` hàng ngàn đơn vị, `bedrooms` chỉ từ 1 – 5) về phân phối chuẩn hóa có trung bình $\mu = 0$ và độ lệch chuẩn $\sigma = 1$.
* **Cạm bẫy sống còn:** Tuyệt đối không gọi `fit` hay `fit_transform` trên tập Test! Việc tính toán $\mu$ và $\sigma$ từ tập Test sẽ làm lộ thông tin phân phối của tương lai vào quá trình huấn luyện (**Data Leakage**).

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

# CHỈ học mean và std trên tập Train
X_train_scaled = scaler.fit_transform(X_train)

# DÙNG LẠI mean và std của Train để biến đổi tập Test
X_test_scaled = scaler.transform(X_test)

print("Kỳ vọng (Mean) trên tập Train (sau chuẩn hóa, xấp xỉ 0):", np.round(X_train_scaled.mean(axis=0)[:5], 4))
print("Độ lệch chuẩn (Std) trên tập Train (sau chuẩn hóa, xấp xỉ 1):", np.round(X_train_scaled.std(axis=0)[:5], 4))


### Khối 1.5: Khởi tạo và huấn luyện mô hình Hồi quy tuyến tính
* **Thực hiện:** Giải phương trình nghiệm giải tích $w = (X^T X)^{-1} X^T y$ để tìm tập trọng số $w$ tối ưu tương ứng với từng đặc trưng.

In [ ]:
from sklearn.linear_model import LinearRegression

lr_model = LinearRegression()
lr_model.fit(X_train_scaled, y_train)

print(f"Hệ số chặn (Intercept w0): {lr_model.intercept_:,.2f} USD")
print(f"Số lượng trọng số đặc trưng (w1 -> wd): {len(lr_model.coef_)}")


### Khối 1.6: Đo lường hiệu năng dự đoán trên tập kiểm thử
* **Thực hiện:** So sánh chênh lệch giữa giá nhà thực tế $y_{test}$ và giá dự đoán $\hat{y}_{pred}$.
* **Ý nghĩa các chỉ số:**
  * **MAE (Mean Absolute Error):** Độ lệch trung bình tính theo đơn vị USD, trực quan và ít bị ảnh hưởng bởi điểm ngoại lai.
  * **RMSE (Root Mean Squared Error):** Phạt nặng các điểm dự đoán sai lệch nghiêm trọng do thành phần bình phương sai số.
  * **$R^2$ (Hệ số xác định):** Đo lường tỷ lệ phần trăm phương sai của giá nhà được mô hình giải thích ($R^2 \in (-\infty, 1]$, càng gần 1 mô hình khớp càng tốt).

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

y_pred = lr_model.predict(X_test_scaled)

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print(f"{'Metric':<10} | {'Giá trị':<20} | {'Đơn vị':<10}")
print("-" * 45)
print(f"{'MAE':<10} | {mae:<20,.2f} | USD")
print(f"{'RMSE':<10} | {rmse:<20,.2f} | USD")
print(f"{'R2 Score':<10} | {r2:<20.4f} | Không thứ nguyên (tỷ lệ)")


### Khối 1.7: Khảo sát phần dư (Residuals Analysis) & Độ quan trọng đặc trưng
* **Thực hiện:** Trực quan hóa phần dư nhằm kiểm chứng các giả định cổ điển của mô hình tuyến tính: phần dư phải phân bố ngẫu nhiên quanh trục 0 (**Homoskedasticity**). Nếu biểu đồ có dạng phễu hoặc đường cong, mô hình đang gặp hiện tượng phương sai thay đổi (**Heteroskedasticity**) hoặc bỏ sót mối quan hệ phi tuyến.
* Đồng thời, trích xuất và vẽ biểu đồ các hệ số hồi quy $w$ để phân tích độ quan trọng và chiều hướng tác động của từng đặc trưng.

In [ ]:
# Tính phần dư
residuals = y_test - y_pred

# 1. Vẽ biểu đồ phân tán phần dư (Residual Plot)
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

sns.scatterplot(x=y_pred, y=residuals, alpha=0.7, ax=axes[0], color='#1f77b4', edgecolor='k', s=60)
axes[0].axhline(0, color='red', linestyle='--', linewidth=2)
axes[0].set_xlabel("Giá trị dự đoán (Predicted Price - USD)", fontsize=11)
axes[0].set_ylabel("Phần dư (Residual = Actual - Predicted)", fontsize=11)
axes[0].set_title("Biểu đồ phân tán phần dư (Residual Plot)", fontsize=12, fontweight='bold')

# 2. Vẽ biểu đồ so sánh Thực tế vs Dự đoán
sns.scatterplot(x=y_test, y=y_pred, alpha=0.7, ax=axes[1], color='#9467bd', edgecolor='k', s=60)
p1 = max(max(y_pred), max(y_test))
p2 = min(min(y_pred), min(y_test))
axes[1].plot([p2, p1], [p2, p1], 'r--', lw=2, label='Đường hoàn hảo y=x')
axes[1].set_xlabel("Giá nhà thực tế (Actual Price - USD)", fontsize=11)
axes[1].set_ylabel("Giá nhà dự đoán (Predicted Price - USD)", fontsize=11)
axes[1].set_title("Thực tế vs Dự đoán (Actual vs Predicted)", fontsize=12, fontweight='bold')
axes[1].legend()

plt.tight_layout()
plt.show()

# 3. Trích xuất và trực quan hóa hệ số hồi quy
coef_series = pd.Series(lr_model.coef_, index=X_encoded.columns).sort_values(ascending=True)

plt.figure(figsize=(10, 6))
colors = ['#d62728' if x < 0 else '#2ca02c' for x in coef_series.values]
bars = coef_series.plot(kind='barh', color=colors)
plt.xlabel("Hệ số hồi quy chuẩn hóa (Standardized Coefficient w - USD)", fontsize=11)
plt.title("Độ quan trọng và chiều hướng ảnh hưởng của các thuộc tính đến Giá nhà", fontsize=12, fontweight='bold')
plt.grid(True, axis='x', linestyle=':', alpha=0.6)

for idx, val in enumerate(coef_series.values):
    offset = 10000 if val >= 0 else -60000
    plt.text(val + offset, idx - 0.15, f"{val:+,.0f}", fontsize=9, fontweight='bold')

plt.tight_layout()
plt.show()

print("\nBảng chi tiết hệ số trọng số hồi quy:")
for k, v in coef_series.sort_values(ascending=False).items():
    print(f"  {k:35s}: {v:+12,.2f} USD")


### Nhận xét và Kết luận Bài toán 1:
1. **Chất lượng mô hình:** Mô hình đạt $R^2 \approx 0.6529$, nghĩa là hơn **65.29%** sự biến thiên của giá nhà được giải thích tốt bởi 13 đặc trưng đầu vào. Sai số tuyệt đối trung bình $MAE \approx 970,043$ USD.
2. **Đặc trưng có ảnh hưởng tích cực lớn nhất:**
   * `bathrooms` (+521,879 USD) và `area` (+519,552 USD) là hai yếu tố đẩy giá nhà tăng mạnh nhất trên mỗi đơn vị độ lệch chuẩn.
   * `airconditioning` (+365,157 USD), `stories` (+349,251 USD) và `prefarea` (+266,656 USD) đóng góp tích cực tiếp theo.
3. **Đặc trưng có ảnh hưởng tiêu cực:**
   * So với mức chuẩn `furnished` (đầy đủ nội thất), việc nhà ở mức `furnishingstatus_semi-furnished` làm giảm khoảng -62,837 USD, và ở mức `furnishingstatus_unfurnished` làm giảm tới -192,016 USD. Kết quả này phản ánh chính xác quy luật kinh tế học bất động sản.
4. **Kiểm tra phần dư:** Đa số các điểm phần dư phân bổ đều quanh đường thẳng $y=0$, tuy nhiên ở dải giá trị dự đoán cao (> 8 triệu USD), phần dư có xu hướng mở rộng (phương sai tăng nhẹ), cho thấy với những căn biệt thự siêu sang có thể cần bổ sung đặc trưng phi tuyến hoặc mô hình phức tạp hơn.
